## MitoEye Validation

In [1]:
# %% [markdown]
# # Multi-fold Segmentation Evaluation Pipeline

# %%
import json
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tifffile
from sklearn.metrics import cohen_kappa_score

# — Ensure pandas will display full DataFrames and string columns —
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

def scan_tiffs(base_dirs: list[Path]) -> pd.DataFrame:
    """
    Scan for .tif/.tiff files across multiple directories and return a DataFrame
    with one row per file: folder, filename, width, height, depth, channels.
    """
    print(f"Scanning TIFFs across {len(base_dirs)} directories...")
    records = []
    
    for base_dir in base_dirs:
        print(f"\nScanning {base_dir.name}...")
        # Only look for non-overlay TIFFs (original files)
        tiff_files = [f for f in base_dir.glob('*.tiff') if '3D_OVERLAY' not in f.name]
        tiff_files.extend([f for f in base_dir.glob('*.tif') if '3D_OVERLAY' not in f.name])
        
        for tif_path in tiff_files:
            print(f"  Reading {tif_path.name}")
            try:
                with tifffile.TiffFile(str(tif_path)) as tif:
                    depth = len(tif.pages)
                    arr0 = tif.pages[0].asarray()
                    H, W = arr0.shape[:2]
                    C = arr0.shape[2] if arr0.ndim == 3 else 1
            except Exception as e:
                print(f"    ❌ Could not read {tif_path.name}: {e}")
                continue
            
            records.append({
                'folder': base_dir.name,
                'filename': tif_path.name,
                'width': W,
                'height': H,
                'depth': depth,
                'channels': C,
            })
    
    df = pd.DataFrame(records)
    print(f"\nScanned {len(df)} TIFF files total.")
    print("\nDisplaying full table:")
    display(df)
    return df


def extract_fold_from_overlay(filename: str) -> int:
    """
    Extract fold number from overlay filename.
    E.g., "B3Gt061_1_A5v2_78-205_combined_3D_OVERLAY_model4005_fold1.tiff" -> 1
    """
    match = re.search(r'fold(\d+)', filename)
    if match:
        return int(match.group(1))
    return None


def compute_metrics_for_splits(
    split_dirs: list[Path],
    out_dir: Path
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    For each split directory, compute SENS, SPEC, ACC, DSC, AVD(%), Kappa(κ)
    by comparing ground truth (red channel of original) with predictions
    (red channel of overlay). Writes out two CSVs under `out_dir`:
      - metrics_per_file_<timestamp>.csv
      - metrics_summary_by_fold_<timestamp>.csv
    Returns (df_all, df_summary).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    all_records = []
    
    for split_dir in split_dirs:
        split_name = split_dir.name
        fold_num = int(split_name.replace('split', ''))
        
        print(f"\n{'='*60}")
        print(f"Processing {split_name} (fold {fold_num})")
        print(f"{'='*60}")
        
        # Find all overlay files in this split
        overlay_files = list(split_dir.glob('*3D_OVERLAY*.tiff'))
        overlay_files.extend(list(split_dir.glob('*3D_OVERLAY*.tif')))
        
        print(f"Found {len(overlay_files)} overlay files.")
        
        for i, overlay_path in enumerate(overlay_files, 1):
            overlay_name = overlay_path.name
            print(f"\n  [{i}/{len(overlay_files)}] Processing: {overlay_name}")
            
            # Find corresponding original file (without overlay suffix)
            # Remove the overlay suffix pattern to get the base name
            base_name = re.sub(r'_3D_OVERLAY_model\d+_fold\d+', '', overlay_path.stem)
            
            # Look for the original file
            orig_path = None
            for ext in ['.tiff', '.tif']:
                candidate = split_dir / f"{base_name}{ext}"
                if candidate.exists():
                    orig_path = candidate
                    break
            
            if orig_path is None:
                print(f"    ❌ Original file not found for {overlay_name}")
                continue
            
            print(f"    Found original: {orig_path.name}")
            
            try:
                # Load original file
                orig_arr = tifffile.imread(str(orig_path))
                print(f"    Original shape: {orig_arr.shape}")
                
                # Extract ground truth from RED channel (channel 0)
                if orig_arr.ndim == 4 and orig_arr.shape[-1] >= 3:
                    # 3D volume with RGB channels
                    gt = orig_arr[..., 0]  # Red channel
                elif orig_arr.ndim == 3 and orig_arr.shape[-1] >= 3:
                    # 2D image with RGB channels
                    gt = orig_arr[..., 0][np.newaxis, ...]  # Red channel, add depth dimension
                elif orig_arr.ndim == 3 and orig_arr.shape[-1] == 1:
                    # 3D volume with single channel
                    gt = orig_arr[..., 0]
                elif orig_arr.ndim == 2:
                    # 2D single channel
                    gt = orig_arr[np.newaxis, ...]
                else:
                    gt = orig_arr
                
                # Load overlay file
                overlay_arr = tifffile.imread(str(overlay_path))
                print(f"    Overlay shape: {overlay_arr.shape}")
                
                # Extract prediction from RED channel (channel 0)
                if overlay_arr.ndim == 4 and overlay_arr.shape[-1] >= 3:
                    # 3D volume with RGB channels
                    pred = overlay_arr[..., 0]  # Red channel
                elif overlay_arr.ndim == 3 and overlay_arr.shape[-1] >= 3:
                    # 2D image with RGB channels
                    pred = overlay_arr[..., 0][np.newaxis, ...]  # Red channel, add depth dimension
                elif overlay_arr.ndim == 3 and overlay_arr.shape[-1] == 1:
                    # 3D volume with single channel
                    pred = overlay_arr[..., 0]
                elif overlay_arr.ndim == 2:
                    # 2D single channel
                    pred = overlay_arr[np.newaxis, ...]
                else:
                    pred = overlay_arr
                
                # Ensure same shape
                if gt.shape != pred.shape:
                    print(f"    ⚠ Shape mismatch: GT {gt.shape} vs Pred {pred.shape}")
                    # Try to match shapes
                    min_depth = min(gt.shape[0], pred.shape[0])
                    gt = gt[:min_depth]
                    pred = pred[:min_depth]
                
                # Normalize and threshold
                gt = gt.astype(np.float32)
                if gt.max() > 1.0:
                    gt /= gt.max()
                gt_mask = gt > 0.5
                
                pred = pred.astype(np.float32)
                if pred.max() > 1.0:
                    pred /= pred.max()
                pred_mask = pred > 0.5
                
                # Compute metrics
                g = gt_mask.ravel()
                p = pred_mask.ravel()
                
                TP = np.logical_and(g, p).sum()
                TN = np.logical_and(~g, ~p).sum()
                FP = np.logical_and(~g, p).sum()
                FN = np.logical_and(g, ~p).sum()
                total = g.size
                
                SENS = round(TP/(TP+FN) if TP+FN > 0 else 0.0, 4)
                SPEC = round(TN/(TN+FP) if TN+FP > 0 else 0.0, 4)
                ACC = round((TP+TN)/total if total > 0 else 0.0, 4)
                DSC = round(2*TP/(2*TP+FP+FN) if (2*TP+FP+FN) > 0 else 0.0, 4)
                AVD = round(abs(g.sum()-p.sum())/g.sum()*100 if g.sum() > 0 else 0.0, 4)
                KAPPA = round(cohen_kappa_score(g, p), 4)
                
                all_records.append({
                    'fold': fold_num,
                    'split': split_name,
                    'filename': overlay_name,
                    'original': orig_path.name,
                    'SENS': SENS,
                    'SPEC': SPEC,
                    'ACC': ACC,
                    'DSC': DSC,
                    'AVD (%)': AVD,
                    'Kappa (κ)': KAPPA
                })
                
                print(f"    ✓ Metrics: SENS={SENS:.3f}, SPEC={SPEC:.3f}, ACC={ACC:.3f}, DSC={DSC:.3f}, AVD={AVD:.2f}%, κ={KAPPA:.3f}")
                
            except Exception as e:
                print(f"    ❌ Error processing files: {e}")
                continue
    
    # Create DataFrame
    df_all = pd.DataFrame(all_records)
    
    if len(df_all) > 0:
        # Table 1: per-file results with overall average
        metrics_cols = ['SENS', 'SPEC', 'ACC', 'DSC', 'AVD (%)', 'Kappa (κ)']
        avg = df_all[metrics_cols].mean().round(4)
        avg_row = {
            'fold': 'Overall',
            'split': 'Average',
            'filename': '-',
            'original': '-',
            **avg.to_dict()
        }
        df_all_with_avg = pd.concat([df_all, pd.DataFrame([avg_row])], ignore_index=True)
        
        # Save per-file metrics
        table1_fp = out_dir / f'metrics_per_file_{ts}.csv'
        df_all_with_avg.to_csv(table1_fp, index=False)
        print(f"\n✅ Saved per-file metrics to {table1_fp}")
        
        # Table 2: per-fold summary
        per_fold = (
            df_all.groupby(['fold', 'split'])[metrics_cols]
            .agg(['mean', 'std', 'count'])
            .round(4)
        )
        
        # Flatten multi-level columns
        per_fold.columns = ['_'.join(col).strip() for col in per_fold.columns.values]
        per_fold = per_fold.reset_index()
        
        # Add overall summary
        overall_stats = {}
        for metric in metrics_cols:
            overall_stats[f'{metric}_mean'] = df_all[metric].mean()
            overall_stats[f'{metric}_std'] = df_all[metric].std()
            overall_stats[f'{metric}_count'] = len(df_all)
        
        overall_row = pd.DataFrame([{
            'fold': 'Overall',
            'split': 'All',
            **overall_stats
        }])
        
        df_summary = pd.concat([per_fold, overall_row], ignore_index=True)
        
        # Save fold summary
        table2_fp = out_dir / f'metrics_summary_by_fold_{ts}.csv'
        df_summary.to_csv(table2_fp, index=False)
        print(f"✅ Saved fold-summary metrics to {table2_fp}")
        
        # Display results
        print("\n" + "="*80)
        print("TABLE 1: Per-file Metrics")
        print("="*80)
        display(df_all_with_avg)
        
        print("\n" + "="*80)
        print("TABLE 2: Summary by Fold")
        print("="*80)
        display(df_summary)
        
        # Create simplified summary table showing just means
        simple_summary = df_all.groupby('fold')[metrics_cols].mean().round(4)
        overall_means = df_all[metrics_cols].mean().round(4)
        overall_means.name = 'Overall'
        simple_summary = pd.concat([simple_summary, overall_means.to_frame().T])
        
        print("\n" + "="*80)
        print("TABLE 3: Simplified Summary (Means Only)")
        print("="*80)
        display(simple_summary)
        
        # Save simplified summary
        table3_fp = out_dir / f'metrics_simple_summary_{ts}.csv'
        simple_summary.to_csv(table3_fp)
        print(f"✅ Saved simplified summary to {table3_fp}")
        
    else:
        print("\n⚠ No valid data processed!")
        df_all_with_avg = df_all
        df_summary = pd.DataFrame()
    
    return df_all_with_avg, df_summary

In [2]:
# Define paths
base_path = Path('/media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL')
split_dirs = [base_path / f'split{i}' for i in range(1, 6)]
out_dir = Path('./evaluation_results')

# Verify all directories exist
for split_dir in split_dirs:
    if not split_dir.exists():
        print(f"⚠ Warning: {split_dir} does not exist!")
    else:
        print(f"✓ Found: {split_dir}")

# Scan TIFFs across all splits (optional, for overview)
print("\n" + "="*80)
print("SCANNING ALL TIFF FILES")
print("="*80)
df_scan = scan_tiffs(split_dirs)

# Compute metrics for all splits
print("\n" + "="*80)
print("COMPUTING METRICS FOR ALL SPLITS")
print("="*80)
df_all, df_summary = compute_metrics_for_splits(split_dirs, out_dir)

print("\n" + "="*80)
print("EVALUATION COMPLETE!")
print("="*80)
print(f"Results saved to: {out_dir}")

✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split1
✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split2
✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split3
✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split4
✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split5

SCANNING ALL TIFF FILES
Scanning TIFFs across 5 directories...

Scanning split1...
  Reading B3Gt061PbV3_A7_converted_381-508_combined.tiff
  Reading B3Gt061_1_A5v2_78-205_combined.tiff
  Reading B3GT101_220825-1-128_combined.tiff
  Reading B3GT101_220920_converted_-1-128_combined.tiff

Scanning split2...
  Reading B3GT084_BB1_170722-a_-1-128_combined.tiff
  Reading B3GT084_Bb2a_170415-a_-13-140_combined.tiff
  Reading B3GT109_reda_170818-a_-30-157_combined.tiff
  Reading B3GT109_redb_170816-a-1-128_combined.tiff

Scanning split3...
  Reading B3Gt029B1_a2_try3_731-858_combined.tiff
  Reading B3Gt029B1_a3b_try4_20-147_combined.tiff
  Reading B3TP1

,folder,filename,width,height,depth,channels
0,split1,B3Gt061PbV3_A7_converted_381-508_combined.tiff,2671,957,128,3
1,split1,B3Gt061_1_A5v2_78-205_combined.tiff,2666,996,128,3
2,split1,B3GT101_220825-1-128_combined.tiff,2665,995,128,3
3,split1,B3GT101_220920_converted_-1-128_combined.tiff,2675,1008,128,3
4,split2,B3GT084_BB1_170722-a_-1-128_combined.tiff,2667,1346,128,3
5,split2,B3GT084_Bb2a_170415-a_-13-140_combined.tiff,2673,1074,128,3
6,split2,B3GT109_reda_170818-a_-30-157_combined.tiff,2950,1010,128,3
7,split2,B3GT109_redb_170816-a-1-128_combined.tiff,2988,1006,128,3
8,split3,B3Gt029B1_a2_try3_731-858_combined.tiff,2064,1138,128,3
9,split3,B3Gt029B1_a3b_try4_20-147_combined.tiff,2329,1003,128,3



COMPUTING METRICS FOR ALL SPLITS

Processing split1 (fold 1)
Found 4 overlay files.

  [1/4] Processing: B3Gt061PbV3_A7_converted_381-508_combined_3D_OVERLAY_model4005_fold0.tiff
    Found original: B3Gt061PbV3_A7_converted_381-508_combined.tiff
    Original shape: (128, 957, 2671, 3)
    Overlay shape: (128, 957, 2671, 3)
    ✓ Metrics: SENS=0.921, SPEC=0.995, ACC=0.988, DSC=0.930, AVD=1.92%, κ=0.924

  [2/4] Processing: B3Gt061_1_A5v2_78-205_combined_3D_OVERLAY_model4005_fold0.tiff
    Found original: B3Gt061_1_A5v2_78-205_combined.tiff
    Original shape: (128, 996, 2666, 3)
    Overlay shape: (128, 996, 2666, 3)
    ✓ Metrics: SENS=0.983, SPEC=0.991, ACC=0.990, DSC=0.952, AVD=6.38%, κ=0.947

  [3/4] Processing: B3GT101_220825-1-128_combined_3D_OVERLAY_model4005_fold0.tiff
    Found original: B3GT101_220825-1-128_combined.tiff
    Original shape: (128, 995, 2665, 3)
    Overlay shape: (128, 995, 2665, 3)
    ✓ Metrics: SENS=0.987, SPEC=0.998, ACC=0.997, DSC=0.979, AVD=1.66%, κ=0.97

,fold,split,filename,original,SENS,SPEC,ACC,DSC,AVD (%),Kappa (κ)
0,1,split1,B3Gt061PbV3_A7_converted_381-508_combined_3D_OVERLAY_model4005_fold0.tiff,B3Gt061PbV3_A7_converted_381-508_combined.tiff,0.9212,0.9945,0.9882,0.9302,1.9218,0.9238
1,1,split1,B3Gt061_1_A5v2_78-205_combined_3D_OVERLAY_model4005_fold0.tiff,B3Gt061_1_A5v2_78-205_combined.tiff,0.9825,0.9911,0.9903,0.9522,6.3805,0.9467
2,1,split1,B3GT101_220825-1-128_combined_3D_OVERLAY_model4005_fold0.tiff,B3GT101_220825-1-128_combined.tiff,0.9867,0.9976,0.9968,0.9786,1.6598,0.9769
3,1,split1,B3GT101_220920_converted_-1-128_combined_3D_OVERLAY_model4005_fold0.tiff,B3GT101_220920_converted_-1-128_combined.tiff,0.9655,0.9918,0.9873,0.9628,0.5740,0.9551
4,2,split2,B3GT084_BB1_170722-a_-1-128_combined_3D_OVERLAY_model4005_fold1.tiff,B3GT084_BB1_170722-a_-1-128_combined.tiff,0.9215,0.9987,0.9913,0.9531,6.6261,0.9483
5,2,split2,B3GT084_Bb2a_170415-a_-13-140_combined_3D_OVERLAY_model4005_fold1.tiff,B3GT084_Bb2a_170415-a_-13-140_combined.tiff,0.7692,0.9995,0.9809,0.8669,22.5439,0.8568
6,2,split2,B3GT109_reda_170818-a_-30-157_combined_3D_OVERLAY_model4005_fold1.tiff,B3GT109_reda_170818-a_-30-157_combined.tiff,0.8971,0.9990,0.9931,0.9377,8.6700,0.9341
7,2,split2,B3GT109_redb_170816-a-1-128_combined_3D_OVERLAY_model4005_fold1.tiff,B3GT109_redb_170816-a-1-128_combined.tiff,0.8618,0.9992,0.9896,0.9206,12.7716,0.9151
8,3,split3,B3Gt029B1_a2_try3_731-858_combined_3D_OVERLAY_model4005_fold2.tiff,B3Gt029B1_a2_try3_731-858_combined.tiff,0.9692,0.9935,0.9907,0.9608,1.7528,0.9555
9,3,split3,B3Gt029B1_a3b_try4_20-147_combined_3D_OVERLAY_model4005_fold2.tiff,B3Gt029B1_a3b_try4_20-147_combined.tiff,0.9505,0.9964,0.9935,0.9485,0.4134,0.9451



TABLE 2: Summary by Fold


,fold,split,SENS_mean,SENS_std,SENS_count,SPEC_mean,SPEC_std,SPEC_count,ACC_mean,ACC_std,ACC_count,DSC_mean,DSC_std,DSC_count,AVD (%)_mean,AVD (%)_std,AVD (%)_count,Kappa (κ)_mean,Kappa (κ)_std,Kappa (κ)_count
0,1,split1,0.9640,0.030000,4,0.993800,0.003000,4,0.990600,0.004300,4,0.956000,0.020300,4,2.6340,2.564900,4,0.95060,0.021900,4
1,2,split2,0.8624,0.066800,4,0.999100,0.000300,4,0.988700,0.005400,4,0.919600,0.037500,4,12.6529,7.071800,4,0.91360,0.040200,4
2,3,split3,0.9715,0.015800,4,0.994900,0.002100,4,0.993000,0.002200,4,0.962200,0.010000,4,1.9341,1.369300,4,0.95830,0.010200,4
3,4,split4,0.8799,0.045000,4,0.996200,0.002500,4,0.981900,0.004600,4,0.920400,0.032400,4,8.8699,4.381700,4,0.91000,0.032000,4
4,5,split5,0.9463,0.035800,4,0.994000,0.006100,4,0.988400,0.009100,4,0.950800,0.022600,4,4.1020,2.772100,4,0.94420,0.028100,4
5,Overall,All,0.9248,0.059421,20,0.995605,0.003601,20,0.988555,0.006267,20,0.941785,0.029952,20,6.0386,5.581266,20,0.93533,0.032351,20



TABLE 3: Simplified Summary (Means Only)


,SENS,SPEC,ACC,DSC,AVD (%),Kappa (κ)
1,0.9640,0.9938,0.9906,0.9560,2.6340,0.9506
2,0.8624,0.9991,0.9887,0.9196,12.6529,0.9136
3,0.9715,0.9949,0.9930,0.9622,1.9341,0.9583
4,0.8799,0.9962,0.9819,0.9204,8.8699,0.9100
5,0.9463,0.9940,0.9884,0.9508,4.1020,0.9442
Overall,0.9248,0.9956,0.9886,0.9418,6.0386,0.9353


✅ Saved simplified summary to evaluation_results/metrics_simple_summary_20250806_164030.csv

EVALUATION COMPLETE!
Results saved to: evaluation_results
